> **Note:** This notebook reuses patterns inspired by the Azure AI Gateway Labs (https://github.com/Azure-Samples/AI-Gateway).

# 🏰 Citadel Backend Contracts - Testing Center

## Explore How to Bring New AI Backends/Models into the Citadel Governance Hub!

Use this Jupyter notebook with Python code snippets to onboard all your AI backends and all associated routing logic, including:
- Configuring LLM backend endpoints and authentication
- Generating Bicep parameter files for deployment
- Deploying backend pools and policy fragments
- Testing deployed models through multiple API formats
- Verifying streaming and SDK integration

> **Note:** This notebook assumes you have already deployed your Citadel Governance Hub. If you haven't done so, please refer to the [Citadel Governance Hub Deployment Guide](../guides/full-deployment-guide.md) or [Citadel Governance Hub Quick Deployment Guide](../guides/quick-deployment-guide.md) before proceeding.

## Azure Prerequisites

- An existing Citadel Governance Hub deployment with APIM configured
- LLM backend endpoints (AI Foundry, Azure OpenAI, or external) provisioned and accessible
- Managed Identity configured for APIM authentication to backend services

<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

Configure the following variables according to your environment before running the notebook:

> **NOTE:** If you used Azure Developer CLI to deploy your AI Governance Hub, you can find the required `llm_backends_config` value by running the following command: `azd env get LLM_BACKENDS_CONFIG`

In [2]:
# Read environment values from azd
import subprocess, json

def azd_get_value(key):
    """Read a value from the current azd environment."""
    result = subprocess.run(
        ["azd", "env", "get-value", key],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"Failed to read {key}: {result.stderr.strip()}")
    return result.stdout.strip()

env_governance_hub_resource_group = azd_get_value("AZURE_RESOURCE_GROUP")
env_location = azd_get_value("AZURE_LOCATION")
env_llm_backends_config = json.loads(azd_get_value("LLM_BACKEND_CONFIG"))

print(f"Citadel Hub resource group: {env_governance_hub_resource_group}")
print(f"Location: {env_location}")
print(f"LLM Backends: {len(env_llm_backends_config)} backend(s) configured")

Citadel Hub resource group: rg-citadel-workshop
Location: swedencentral
LLM Backends: 2 backend(s) configured


In [3]:
import os
import sys, json, requests, time
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils
from apimtools import APIMClientTool

inference_api_version = "2024-05-01-preview"

# ============================================================================
# REQUIRED: Update these values for your environment
# ============================================================================
governance_hub_resource_group = env_governance_hub_resource_group     # Resource group of the deployed Citadel Governance Hub
location                      = env_location                          # Azure region (e.g. "swedencentral", "eastus")

#### WARNING: UPDATE THIS CONFIGURATION BASED ON YOUR DEPLOYED BACKENDS ####
#### If you used azd to deploy backends as well, you can copy the current backend configuration using `azd env get-value LLM_BACKEND_CONFIG` and paste it here, then update the values as needed. ####
llm_backends_config = env_llm_backends_config

# ============================================================================
# OPTIONAL: Model Aliases
# Group multiple models under a single client-facing name. Clients send the
# alias as the `model` value; the gateway resolves it to a real model.
#   strategy: 'priority' (first available wins) | 'weighted' (round-robin by weights)
# Example:
#   model_aliases = [
#     { "name": "gpt-advanced", "models": ["gpt-5", "gpt-4.1", "gpt-4o"], "strategy": "priority" }
#   ]
# ============================================================================
model_aliases = []

# ============================================================================
# OPTIONAL: Key Vault for backend credential references
# Required when any backend's authConfig uses `keyVaultSecretUri`. The APIM
# managed identity must have `Key Vault Secrets User` role on this vault.
# ============================================================================
key_vault_name = ""

# ============================================================================
# OPTIONAL: AWS credentials (only required when an aws-bedrock backend is used)
# Stored as APIM secret named values: aws-access-key / aws-secret-key / aws-region
# ============================================================================
aws_access_key = ""
aws_secret_key = ""
aws_region = ""

# Managed Identity for APIM authentication (will be auto-discovered if not specified)
apim_managed_identity_name = ""  # Leave empty to auto-discover

utils.print_info(f"Initialization completed.")


👉🏽 Initialization completed.


<a id='1'></a>
### 1️⃣ Verify Azure CLI and Connected Subscription

Ensure Azure CLI is authenticated and connected to the correct subscription:

In [4]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 15:48:17.464311 :2s]
👉🏽 Current user: sofiedelaet@MngEnvMCAP425328.onmicrosoft.com
👉🏽 Tenant ID: 97005946-1354-4b6a-9a5f-a84c67213bee
👉🏽 Subscription ID: b881797f-b05b-4743-8516-17a967f31841


<a id='2'></a>
### 2️⃣ Initialize APIM Client Tool

👉 An existing Citadel Governance Hub deployment is expected. Initialize the APIM client to interact with your deployment:

In [5]:
try:
    apimClientTool = APIMClientTool(
        governance_hub_resource_group
    )
    apimClientTool.initialize()
    
    apim_resource_name = apimClientTool.apim_resource_name
    apim_resource_gateway_url = str(apimClientTool.apim_resource_gateway_url)
    
    utils.print_ok(f"APIM Client Tool initialized successfully!")
    utils.print_info(f"APIM Resource Name: {apim_resource_name}")
    utils.print_info(f"APIM Gateway URL: {apim_resource_gateway_url}")
    
except Exception as e:
    utils.print_error(f"Error initializing APIM Client Tool: {e}")

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 15:48:29.779219 :1s]
👉🏽 Current user: sofiedelaet@MngEnvMCAP425328.onmicrosoft.com
👉🏽 Tenant ID: 97005946-1354-4b6a-9a5f-a84c67213bee
👉🏽 Subscription ID: b881797f-b05b-4743-8516-17a967f31841
⚙️ Running: az resource list -g rg-citadel-workshop --resource-type Microsoft.ApiManagement/service 
✅ Listing APIM Resources ⌚ 15:48:34.104105 :4s]
👉🏽 APIM Service Id: /subscriptions/b881797f-b05b-4743-8516-17a967f31841/resourceGroups/rg-citadel-workshop/providers/Microsoft.ApiManagement/service/apim-6dm44zbv4ms4s
👉🏽 APIM Gateway URL: https://apim-6dm44zbv4ms4s.azure-api.net
👉🏽 Retrieved key 0 for subscription: master
👉🏽 Retrieved key 1 for subscription: LLM-Testing-UniversalLLMAllModels-DEV-SUB-01
👉🏽 Retrieved key 2 for subscription: LLM-Sales-Assistant-DEV-SUB-01
👉🏽 Retrieved key 3 for subscription: LLM-HR-ChatAgent-DEV-SUB-01
👉🏽 Retrieved key 4 for subscription: LLM-Support-Bot-DEV-SUB-01
👉🏽 Retrieved key 5 for subscription: LLM-HR-PIIMaskin

<a id='3'></a>
### 3️⃣ Extract Current APIM Backend-Pools Configuration

Retrieve and analyze the existing backend pools and backends configured in your APIM instance:

In [6]:
# Extract current backends from APIM using the SDK
utils.print_info("Extracting current APIM backends configuration...")

try:
    # Use the APIMClientTool's new get_backends method (uses Azure SDK instead of CLI)
    existing_backends, existing_backend_pools = apimClientTool.get_backends()
    
except Exception as e:
    utils.print_error(f"Error extracting backends: {e}")
    existing_backends = []
    existing_backend_pools = []

👉🏽 Extracting current APIM backends configuration...
👉🏽 Retrieving APIM backends using Azure REST API...
👉🏽 🔗 Backend: aif-6dm44zbv4ms4s-0 -> https://aif-6dm44zbv4ms4s-0.cognitiveservices.azure.com/
👉🏽 🔗 Backend: aif-6dm44zbv4ms4s-1 -> https://aif-6dm44zbv4ms4s-1.cognitiveservices.azure.com/
👉🏽 🔗 Backend: content-safety-backend -> https://aif-6dm44zbv4ms4s-0.cognitiveservices.azure.com/
👉🏽 📦 Backend Pool: DeepSeek-R1-backend-pool (2 backends)
👉🏽 🔗 Backend: foundry-embeddings -> https://aif-6dm44zbv4ms4s-0.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings
👉🏽 📦 Backend Pool: gpt-54-mini-backend-pool (2 backends)
👉🏽 🔗 Backend: ms-learn-mcp-server -> https://learn.microsoft.com/api/mcp
👉🏽 📦 Backend Pool: Phi-4-backend-pool (2 backends)
👉🏽 📦 Backend Pool: text-embedding-3-large-backend-pool (2 backends)
✅ Found 5 individual backends and 4 backend pools ⌚ 15:49:21.737182 


In [7]:
# Get supported models from the policy fragment (if exists)
try:
    supported_models_from_policy = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_ok(f"Supported models in APIM policy fragment 'set-backend-pools':")
    for model in supported_models_from_policy:
        print(f"  • {model}")
except Exception as e:
    utils.print_warning(f"Could not retrieve policy fragment (may not exist yet): {e}")
    supported_models_from_policy = []

👉🏽 Retrieved policy fragment: set-backend-pools
👉🏽 Found 7 unique supported models
✅ Supported models in APIM policy fragment 'set-backend-pools': ⌚ 15:50:47.344422 
  • DeepSeek-R1
  • Mistral-Large-3
  • Phi-4
  • gpt-4.1
  • gpt-5.2
  • gpt-5.4-mini
  • text-embedding-3-large


In [8]:
# Display summary of current configuration
utils.print_info("\n" + "="*60)
utils.print_info("CURRENT APIM BACKEND CONFIGURATION SUMMARY")
utils.print_info("="*60)

if existing_backends:
    print("\n📋 Individual Backends:")
    for backend in existing_backends:
        print(f"  • {backend['name']}")
        print(f"    URL: {backend['url']}")
        if backend['supportedModels']:
            print(f"    Models: {', '.join(backend['supportedModels'])}")

if existing_backend_pools:
    print("\n📦 Backend Pools:")
    for pool in existing_backend_pools:
        print(f"  • {pool['name']}")
        for svc in pool['services']:
            print(f"    - {svc.get('id', 'N/A')} (priority: {svc.get('priority', 'N/A')}, weight: {svc.get('weight', 'N/A')})")

if supported_models_from_policy:
    print(f"\n🤖 Total Supported Models: {len(supported_models_from_policy)}")
    print(f"   {', '.join(supported_models_from_policy)}")

👉🏽 
👉🏽 CURRENT APIM BACKEND CONFIGURATION SUMMARY
👉🏽 ============================================================

📋 Individual Backends:
  • aif-6dm44zbv4ms4s-0
    URL: https://aif-6dm44zbv4ms4s-0.cognitiveservices.azure.com/
    Models: gpt-4.1, DeepSeek-R1, text-embedding-3-large, Mistral-Large-3, gpt-5.4-mini, Phi-4
  • aif-6dm44zbv4ms4s-1
    URL: https://aif-6dm44zbv4ms4s-1.cognitiveservices.azure.com/
    Models: Phi-4, gpt-5.4-mini, gpt-5.2, DeepSeek-R1, text-embedding-3-large
  • content-safety-backend
    URL: https://aif-6dm44zbv4ms4s-0.cognitiveservices.azure.com/
  • foundry-embeddings
    URL: https://aif-6dm44zbv4ms4s-0.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings
  • ms-learn-mcp-server
    URL: https://learn.microsoft.com/api/mcp

📦 Backend Pools:
  • DeepSeek-R1-backend-pool
    - /subscriptions/b881797f-b05b-4743-8516-17a967f31841/resourceGroups/rg-citadel-workshop/providers/Microsoft.ApiManagement/service/apim-6dm44zbv4ms4s/backe

<a id='4'></a>
### 4️⃣ Discover Managed Identity for APIM Authentication

Auto-discover or specify the user-assigned managed identity used by APIM:

In [9]:
# Discover managed identity from APIM using the SDK
utils.print_info("Discovering managed identity configuration...")

# Use the APIMClientTool's get_managed_identity_info method
managed_identity_info = apimClientTool.get_managed_identity_info()

managed_identity_client_id = managed_identity_info.get('clientId')
managed_identity_name = managed_identity_info.get('name') or apim_managed_identity_name
managed_identity_resource_group = managed_identity_info.get('resourceGroup') or governance_hub_resource_group

if not managed_identity_client_id:
    utils.print_warning("Could not auto-discover managed identity. Please specify it manually in the configuration.")
else:
    utils.print_info(f"Client ID: {managed_identity_client_id}")

if managed_identity_name:
    utils.print_ok(f"Managed Identity Name: {managed_identity_name}")
    utils.print_ok(f"Managed Identity Resource Group: {managed_identity_resource_group}")

👉🏽 Discovering managed identity configuration...
✅ Found managed identity client ID in named values: 49b15419... ⌚ 16:05:24.283725 
✅ Found user-assigned managed identity: id-apim-6dm44zbv4ms4s ⌚ 16:05:24.284706 
👉🏽 Client ID: 49b15419-95ad-456d-80f3-338aadd6b46a
✅ Managed Identity Name: id-apim-6dm44zbv4ms4s ⌚ 16:05:24.285789 
✅ Managed Identity Resource Group: rg-citadel-workshop ⌚ 16:05:24.285838 


<a id='5'></a>
### 5️⃣ Generate LLM Backend Parameter File

Generate a customizable `.bicepparam` file with the full list of LLM backends to be integrated with APIM:

In [10]:
# Configure the LLM backends for deployment
# You can modify the llm_backends_config list defined in the initialization cell

utils.print_info("LLM Backends to be deployed:")
for backend in llm_backends_config:
    print(f"\n  🔗 {backend['backendId']}")
    print(f"     Type: {backend['backendType']}")
    print(f"     Endpoint: {backend['endpoint']}")
    auth = backend.get('authType') or backend.get('authScheme') or 'unspecified'
    print(f"     Auth: {auth}")
    auth_config = backend.get('authConfig')
    if auth_config:
        nv = auth_config.get('namedValueKey', 'N/A')
        if auth_config.get('keyVaultSecretUri'):
            print(f"     Auth Source: Key Vault → {auth_config['keyVaultSecretUri']} (named value: {nv})")
        elif auth_config.get('secretValue'):
            print(f"     Auth Source: inline secret (named value: {nv}) ⚠️ testing only")
        else:
            print(f"     Auth Source: named value '{nv}'")
    print(f"     Priority: {backend.get('priority', 1)}, Weight: {backend.get('weight', 100)}")
    # Display supported models with their per-model metadata
    print(f"     Models ({len(backend['supportedModels'])}):")
    for model in backend['supportedModels']:
        model_name = model['name'] if isinstance(model, dict) else model
        if isinstance(model, dict):
            sku = model.get('sku', 'Standard')
            capacity = model.get('capacity', 100)
            fmt = model.get('modelFormat', 'OpenAI')
            ver = model.get('modelVersion', '1')
            extras = []
            if 'retirementDate' in model:
                extras.append(f"Retirement: {model['retirementDate']}")
            if 'apiVersion' in model:
                extras.append(f"API: {model['apiVersion']}")
            if 'timeout' in model:
                extras.append(f"Timeout: {model['timeout']}s")
            if 'inferenceApiVersion' in model:
                extras.append(f"Inference API: {model['inferenceApiVersion']}")
            extra_str = f", {', '.join(extras)}" if extras else ""
            print(f"       - {model_name} (SKU: {sku}, Capacity: {capacity}, Format: {fmt}, Version: {ver}{extra_str})")
        else:
            print(f"       - {model_name}")

# Display model aliases if defined
if model_aliases:
    utils.print_info(f"\nModel Aliases ({len(model_aliases)}):")
    for alias in model_aliases:
        strategy = alias.get('strategy', 'priority')
        models_str = ', '.join(alias.get('models', []))
        weights = alias.get('weights')
        weights_str = f" weights={weights}" if weights else ""
        print(f"  • {alias['name']} → [{models_str}]  (strategy: {strategy}{weights_str})")


👉🏽 LLM Backends to be deployed:

  🔗 aif-6dm44zbv4ms4s-0
     Type: ai-foundry
     Endpoint: https://aif-6dm44zbv4ms4s-0.cognitiveservices.azure.com/
     Auth: managedIdentity
     Priority: 1, Weight: 100
     Models (6):
       - gpt-4.1 (SKU: GlobalStandard, Capacity: 100, Format: OpenAI, Version: 2025-04-14, Retirement: 2026-10-14, API: 2025-04-01-preview, Timeout: 180s)
       - DeepSeek-R1 (SKU: GlobalStandard, Capacity: 1, Format: DeepSeek, Version: 1, Retirement: 2099-12-30, Inference API: 2024-05-01-preview)
       - text-embedding-3-large (SKU: GlobalStandard, Capacity: 100, Format: OpenAI, Version: 1, Retirement: 2027-04-14)
       - Mistral-Large-3 (SKU: GlobalStandard, Capacity: 100, Format: Mistral AI, Version: 1, Retirement: 2099-12-30)
       - gpt-5.4-mini (SKU: GlobalStandard, Capacity: 100, Format: OpenAI, Version: 2026-03-17, Retirement: 2026-09-30)
       - Phi-4 (SKU: GlobalStandard, Capacity: 1, Format: Microsoft, Version: 7, Retirement: 2099-10-14, API: 2025-0

In [11]:
# Generate the .bicepparam file content
bicep_dir = "../bicep/infra/llm-backend-onboarding"
params_file = os.path.join(bicep_dir, "llm-backends-generated-local.bicepparam")

# Format a single model object for Bicep
def format_model_for_bicep(model):
    """Format a model object for Bicep with per-model metadata including optional routing attributes."""
    if isinstance(model, str):
        # Legacy format: just model name string - convert to object with defaults
        return f"{{ name: '{model}' }}"
    
    # New format: model object with metadata
    parts = [f"name: '{model['name']}'"]
    if 'sku' in model:
        parts.append(f"sku: '{model['sku']}'")
    if 'capacity' in model:
        parts.append(f"capacity: {model['capacity']}")
    if 'modelFormat' in model:
        parts.append(f"modelFormat: '{model['modelFormat']}'")
    if 'modelVersion' in model:
        parts.append(f"modelVersion: '{model['modelVersion']}'")
    if 'retirementDate' in model:
        parts.append(f"retirementDate: '{model['retirementDate']}'")
    if 'apiVersion' in model:
        parts.append(f"apiVersion: '{model['apiVersion']}'")
    if 'timeout' in model:
        parts.append(f"timeout: {model['timeout']}")
    if 'inferenceApiVersion' in model:
        parts.append(f"inferenceApiVersion: '{model['inferenceApiVersion']}'")
    
    return "{ " + ", ".join(parts) + " }"

# Format an authConfig object for Bicep
def format_auth_config_for_bicep(auth_config):
    parts = []
    if 'namedValueKey' in auth_config:
        parts.append(f"namedValueKey: '{auth_config['namedValueKey']}'")
    if 'keyVaultSecretUri' in auth_config:
        parts.append(f"keyVaultSecretUri: '{auth_config['keyVaultSecretUri']}'")
    if 'secretValue' in auth_config:
        parts.append(f"secretValue: '{auth_config['secretValue']}'")
    return "{ " + ", ".join(parts) + " }"

# Format backends array for Bicep (uses per-model metadata)
def format_backend_for_bicep(backend):
    """Format a backend configuration for Bicep with per-model metadata."""
    # Format each model with its individual metadata
    models_formatted = [format_model_for_bicep(m) for m in backend['supportedModels']]
    models_str = "\n      ".join(models_formatted)

    # Auth: prefer new authType, fall back to legacy authScheme
    auth_lines = []
    if 'authType' in backend:
        auth_lines.append(f"    authType: '{backend['authType']}'")
    elif 'authScheme' in backend:
        auth_lines.append(f"    authScheme: '{backend['authScheme']}'")
    if 'authConfig' in backend:
        auth_lines.append(f"    authConfig: {format_auth_config_for_bicep(backend['authConfig'])}")
    auth_block = ("\n".join(auth_lines) + "\n") if auth_lines else ""

    return f"""  {{
    backendId: '{backend['backendId']}'
    backendType: '{backend['backendType']}'
    endpoint: '{backend['endpoint']}'
{auth_block}    supportedModels: [
      {models_str}
    ]
    priority: {backend.get('priority', 1)}
    weight: {backend.get('weight', 100)}
  }}"""

# Format model alias array for Bicep
def format_alias_for_bicep(alias):
    parts = [f"name: '{alias['name']}'"]
    models_arr = ", ".join([f"'{m}'" for m in alias.get('models', [])])
    parts.append(f"models: [ {models_arr} ]")
    if 'strategy' in alias:
        parts.append(f"strategy: '{alias['strategy']}'")
    if 'weights' in alias:
        weights_arr = ", ".join([str(w) for w in alias['weights']])
        parts.append(f"weights: [ {weights_arr} ]")
    return "  { " + ", ".join(parts) + " }"

backends_bicep_str = "\n".join([format_backend_for_bicep(b) for b in llm_backends_config])

# Optional sections
aliases_block = ""
if model_aliases:
    aliases_str = "\n".join([format_alias_for_bicep(a) for a in model_aliases])
    aliases_block = f"""

// ============================================================================
// Model Aliases — group multiple models under a single client-facing name
// ============================================================================
param modelAliases = [
{aliases_str}
]
"""
else:
    aliases_block = "\n\nparam modelAliases = []\n"

key_vault_block = ""
if key_vault_name:
    key_vault_block = f"""
// ============================================================================
// Key Vault for backend credential references (used by authConfig.keyVaultSecretUri)
// ============================================================================
param keyVaultName = '{key_vault_name}'
"""

aws_block = ""
if aws_access_key or aws_secret_key or aws_region:
    aws_block = f"""
// ============================================================================
// AWS credentials for Amazon Bedrock backends (stored as APIM secret named values)
// ============================================================================
param awsAccessKey = '{aws_access_key}'
param awsSecretKey = '{aws_secret_key}'
param awsRegion = '{aws_region}'
"""

params_content = f"""using './main.bicep'

// ============================================================================
// LLM Backend Onboarding - Generated Parameter File
// Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}
// ============================================================================

// ============================================================================
// API Management (APIM) Configuration
// ============================================================================
param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{apim_resource_name}'
}}

// ============================================================================
// APIM Managed Identity Configuration
// ============================================================================
param apimManagedIdentity = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{managed_identity_resource_group}'
  name: '{managed_identity_name}'
}}

// ============================================================================
// LLM Backend Configuration Array
// Each backend uses 'authType' (managed-identity | aws-sigv4 | api-key-bearer
//   | api-key-header | none); api-key auth types also require 'authConfig'.
// Each model in supportedModels has its own metadata (sku, capacity,
//   modelFormat, modelVersion, retirementDate, apiVersion, timeout,
//   inferenceApiVersion).
// ============================================================================
param llmBackendConfig = [
{backends_bicep_str}
]

// ============================================================================
// Circuit Breaker Configuration
// ============================================================================
param configureCircuitBreaker = true
{aliases_block}{key_vault_block}{aws_block}"""

# Write the parameter file
utils.print_info(f"Generating parameter file: {params_file}")
with open(params_file, 'w') as f:
    f.write(params_content)

utils.print_ok(f"Parameter file generated successfully!")
print("\n" + "="*60)
print("GENERATED PARAMETER FILE CONTENT:")
print("="*60)
print(params_content)


👉🏽 Generating parameter file: ../bicep/infra/llm-backend-onboarding\llm-backends-generated-local.bicepparam
✅ Parameter file generated successfully! ⌚ 16:24:03.489504 

GENERATED PARAMETER FILE CONTENT:
using './main.bicep'

// ============================================================================
// LLM Backend Onboarding - Generated Parameter File
// Generated: 2026-06-04 16:24:03
// ============================================================================

// ============================================================================
// API Management (APIM) Configuration
// ============================================================================
param apim = {
  subscriptionId: 'b881797f-b05b-4743-8516-17a967f31841'
  resourceGroupName: 'rg-citadel-workshop'
  name: 'apim-6dm44zbv4ms4s'
}

// ============================================================================
// APIM Managed Identity Configuration
// ===========================================================

<a id='6'></a>
### 6️⃣ Deploy LLM Backend Onboarding Bicep

Deploy the LLM backends, backend pools, and policy fragments to APIM:

In [12]:
# Deploy the LLM backend onboarding
deployment_name = f"llm-backend-onboarding-{time.strftime('%Y%m%d%H%M%S')}"
template_file = os.path.join(bicep_dir, "main.bicep")

utils.print_info(f"Starting deployment: {deployment_name}")
utils.print_info(f"Template: {template_file}")
utils.print_info(f"Parameters: {params_file}")

# Run the subscription-level deployment
deployment_cmd = f"az deployment sub create --name {deployment_name} --location {location} --template-file {template_file} --parameters {params_file}"

output = utils.run(
    deployment_cmd,
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed"
)

if output.success:
    utils.print_ok("Deployment completed successfully!")
    
    # Display deployment outputs if available
    outputs = output.json_data.get('properties', {}).get('outputs', {}) if output.json_data else {}
    
    if outputs:
        print("\n" + "="*60)
        print("DEPLOYMENT OUTPUTS:")
        print("="*60)
        
        for key, value in outputs.items():
            print(f"  {key}: {value.get('value')}")
    else:
        utils.print_info("No deployment outputs returned.")
else:
    utils.print_error("Deployment failed. Check the error messages above.")

👉🏽 Starting deployment: llm-backend-onboarding-20260604162647
👉🏽 Template: ../bicep/infra/llm-backend-onboarding\main.bicep
👉🏽 Parameters: ../bicep/infra/llm-backend-onboarding\llm-backends-generated-local.bicepparam
⚙️ Running: az deployment sub create --name llm-backend-onboarding-20260604162647 --location swedencentral --template-file ../bicep/infra/llm-backend-onboarding\main.bicep --parameters ../bicep/infra/llm-backend-onboarding\llm-backends-generated-local.bicepparam 
✅ Deployment 'llm-backend-onboarding-20260604162647' succeeded ⌚ 16:28:20.448067 :32s]
✅ Deployment completed successfully! ⌚ 16:28:20.449671 

DEPLOYMENT OUTPUTS:
  apimGatewayUrl: https://apim-6dm44zbv4ms4s.azure-api.net
  apimServiceName: apim-6dm44zbv4ms4s
  backendIds: ['aif-6dm44zbv4ms4s-0', 'aif-6dm44zbv4ms4s-1']
  modelToBackendMap: {'Mistral-Large-3': 'aif-6dm44zbv4ms4s-0', 'gpt-4.1': 'aif-6dm44zbv4ms4s-0', 'gpt-5.2': 'aif-6dm44zbv4ms4s-1'}
  modelToPoolMap: {'DeepSeek-R1': 'DeepSeek-R1-backend-pool', 'Ph

<a id='7'></a>
### 7️⃣ Verify Deployed Configuration

Verify that the backends, pools, and policy fragments were created successfully:

In [13]:
# Re-initialize APIM client to pick up new backends
apimClientTool.initialize()

# Get updated supported models from policy fragment
try:
    updated_supported_models = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_ok(f"Updated supported models in APIM policy fragment 'set-backend-pools':")
    for model in updated_supported_models:
        print(f"  • {model}")
except Exception as e:
    utils.print_error(f"Error retrieving policy fragment: {e}")
    updated_supported_models = []

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 16:33:02.032409 :2s]
👉🏽 Current user: sofiedelaet@MngEnvMCAP425328.onmicrosoft.com
👉🏽 Tenant ID: 97005946-1354-4b6a-9a5f-a84c67213bee
👉🏽 Subscription ID: b881797f-b05b-4743-8516-17a967f31841
👉🏽 APIM Service Id: /subscriptions/b881797f-b05b-4743-8516-17a967f31841/resourceGroups/rg-citadel-workshop/providers/Microsoft.ApiManagement/service/apim-6dm44zbv4ms4s
👉🏽 APIM Gateway URL: https://apim-6dm44zbv4ms4s.azure-api.net
👉🏽 Retrieved key 0 for subscription: master
👉🏽 Retrieved key 1 for subscription: LLM-Testing-UniversalLLMAllModels-DEV-SUB-01
👉🏽 Retrieved key 2 for subscription: LLM-Sales-Assistant-DEV-SUB-01
👉🏽 Retrieved key 3 for subscription: LLM-HR-ChatAgent-DEV-SUB-01
👉🏽 Retrieved key 4 for subscription: LLM-Support-Bot-DEV-SUB-01
👉🏽 Retrieved key 5 for subscription: LLM-HR-PIIMasking-DEV-SUB-01
👉🏽 Retrieved key 6 for subscription: LLM-Compliance-PIIBlocking-DEV-SUB-01
👉🏽 Retrieved key 7 for subscription: LLM-HR-PIIAnalytics-DEV-

In [14]:
# Display summary of current configuration
utils.print_info("\n" + "="*60)
utils.print_info("CURRENT APIM BACKEND CONFIGURATION SUMMARY")
utils.print_info("="*60)

if existing_backends:
    print("\n📋 Individual Backends:")
    for backend in existing_backends:
        print(f"  • {backend['name']}")
        print(f"    URL: {backend['url']}")
        if backend['supportedModels']:
            print(f"    Models: {', '.join(backend['supportedModels'])}")

if existing_backend_pools:
    print("\n📦 Backend Pools:")
    for pool in existing_backend_pools:
        print(f"  • {pool['name']}")
        for svc in pool['services']:
            print(f"    - {svc.get('id', 'N/A')} (priority: {svc.get('priority', 'N/A')}, weight: {svc.get('weight', 'N/A')})")

if supported_models_from_policy:
    print(f"\n🤖 Total Supported Models: {len(supported_models_from_policy)}")
    print(f"   {', '.join(supported_models_from_policy)}")

👉🏽 
👉🏽 CURRENT APIM BACKEND CONFIGURATION SUMMARY
👉🏽 ============================================================

📋 Individual Backends:
  • aif-6dm44zbv4ms4s-0
    URL: https://aif-6dm44zbv4ms4s-0.cognitiveservices.azure.com/
    Models: gpt-4.1, DeepSeek-R1, text-embedding-3-large, Mistral-Large-3, gpt-5.4-mini, Phi-4
  • aif-6dm44zbv4ms4s-1
    URL: https://aif-6dm44zbv4ms4s-1.cognitiveservices.azure.com/
    Models: Phi-4, gpt-5.4-mini, gpt-5.2, DeepSeek-R1, text-embedding-3-large
  • content-safety-backend
    URL: https://aif-6dm44zbv4ms4s-0.cognitiveservices.azure.com/
  • foundry-embeddings
    URL: https://aif-6dm44zbv4ms4s-0.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings
  • ms-learn-mcp-server
    URL: https://learn.microsoft.com/api/mcp

📦 Backend Pools:
  • DeepSeek-R1-backend-pool
    - /subscriptions/b881797f-b05b-4743-8516-17a967f31841/resourceGroups/rg-citadel-workshop/providers/Microsoft.ApiManagement/service/apim-6dm44zbv4ms4s/backe

<a id='8'></a>
### 8️⃣ Verify Get Available Models Policy Fragment

Verify that the new `get-available-models` policy fragment was created successfully:

In [15]:
# Verify the get-available-models policy fragment exists
try:
    # Use the Azure SDK to check if the policy fragment exists
    policy_fragment = apimClientTool.client.policy_fragment.get(
        resource_group_name=apimClientTool.resource_group_name,
        service_name=apimClientTool.apim_resource_name,
        id="get-available-models"
    )
    
    if policy_fragment:
        utils.print_ok("Policy fragment 'get-available-models' exists!")
        utils.print_info("This fragment returns available model deployments in a format similar to Azure Cognitive Services API.")
        utils.print_info(f"Description: {policy_fragment.description}")
except Exception as e:
    utils.print_warning(f"Could not retrieve 'get-available-models' policy fragment: {e}")
    utils.print_info("This fragment will be created after running the deployment.")

✅ Policy fragment 'get-available-models' exists! ⌚ 16:34:00.732771 
👉🏽 This fragment returns available model deployments in a format similar to Azure Cognitive Services API.
👉🏽 Description: Returns a JSON response listing all available model deployments with their capabilities


<a id='test-foundry'></a>
### 🧪 Test GET /deployments (Microsoft Foundry Integration)

Test the `GET /deployments` endpoint which leverages the `get-available-models` policy fragment. This endpoint is used by Microsoft Foundry to discover available model deployments:

In [16]:
# Test GET /deployments endpoint (Microsoft Foundry integration)
# This endpoint uses the get-available-models policy fragment to return available model deployments
# Testing both Universal LLM API and Azure OpenAI API endpoints

def test_get_deployments(base_endpoint, api_name, api_key):
    """Test GET /deployments for a given API endpoint."""
    deployments_url = f"{base_endpoint}deployments?api-version={inference_api_version}"
    utils.print_info(f"\nTesting {api_name} GET /deployments: {deployments_url}")
    
    try:
        response = requests.get(
            deployments_url,
            headers={"api-key": api_key},
            timeout=30
        )
        
        utils.print_response_code(response)
        
        if response.status_code == 200:
            data = response.json()
            deployments = data.get("value", [])
            
            utils.print_ok(f"{api_name} GET /deployments returned {len(deployments)} model deployment(s)")
            
            print("\n" + "="*60)
            print(f"AVAILABLE MODEL DEPLOYMENTS ({api_name})")
            print("="*60)
            
            for deployment in deployments:
                print(f"\n📦 Deployment: {deployment.get('name', 'N/A')}")
                print(f"   ID: {deployment.get('id', 'N/A')}")
                print(f"   Type: {deployment.get('type', 'N/A')}")
                
                sku = deployment.get('sku', {})
                if sku:
                    print(f"   SKU: {sku.get('name', 'N/A')} (Capacity: {sku.get('capacity', 'N/A')})")
                
                props = deployment.get('properties', {})
                if props:
                    model = props.get('model', {})
                    if model:
                        print(f"   Model: {model.get('name', 'N/A')} (Format: {model.get('format', 'N/A')}, Version: {model.get('version', 'N/A')})")
                    
                    capabilities = props.get('capabilities', {})
                    if capabilities:
                        caps_list = [k for k, v in capabilities.items() if v == 'true' or v == True]
                        print(f"   Capabilities: {', '.join(caps_list) if caps_list else 'N/A'}")
                    
                    print(f"   Status: {props.get('provisioningState', 'N/A')}")
            
            print("\n" + "="*60)
            utils.print_ok(f"{api_name} GET /deployments test completed successfully!")
            return True
        else:
            utils.print_error(f"{api_name} GET /deployments failed: {response.status_code}")
            print(f"Response: {response.text}")
            return False
            
    except Exception as e:
        utils.print_error(f"{api_name} GET /deployments test failed: {str(e)}")
        return False

# Get an API key from subscriptions
if apimClientTool.apim_subscriptions:
    api_key = apimClientTool.apim_subscriptions[0].get("key")
    utils.print_ok(f"Using subscription: {apimClientTool.apim_subscriptions[0].get('name')}")
else:
    utils.print_error("No APIM subscriptions found. Please create a subscription first.")
    api_key = None

if api_key:
    # Test 1: Universal LLM API - GET /models/deployments
    apimClientTool.discover_api("models")
    azure_endpoint_models = str(apimClientTool.azure_endpoint)
    utils.print_info(f"Universal LLM API Base Endpoint: {azure_endpoint_models}models")
    test_get_deployments(f"{azure_endpoint_models}models/", "Universal LLM API", api_key)
    
    # Test 2: Azure OpenAI API - GET /openai/deployments
    try:
        apimClientTool.discover_api("openai")
        azure_endpoint_openai = str(apimClientTool.azure_endpoint)
        utils.print_info(f"\nAzure OpenAI API Base Endpoint: {azure_endpoint_openai}openai")
        test_get_deployments(f"{azure_endpoint_openai}openai/", "Azure OpenAI API", api_key)
    except Exception as e:
        utils.print_warning(f"Azure OpenAI API not found in APIM: {e}")
else:
    utils.print_warning("Cannot test GET /deployments - missing API key")

✅ Using subscription: master ⌚ 16:34:54.344112 
👉🏽 Found API with id /subscriptions/b881797f-b05b-4743-8516-17a967f31841/resourceGroups/rg-citadel-workshop/providers/Microsoft.ApiManagement/service/apim-6dm44zbv4ms4s/apis/universal-llm-api and path models
👉🏽 Azure Endpoint with APIM https://apim-6dm44zbv4ms4s.azure-api.net/
👉🏽 Universal LLM API Base Endpoint: https://apim-6dm44zbv4ms4s.azure-api.net/models
👉🏽 
Testing Universal LLM API GET /deployments: https://apim-6dm44zbv4ms4s.azure-api.net/models/deployments?api-version=2024-05-01-preview
Response status: 200 - OK
✅ Universal LLM API GET /deployments returned 7 model deployment(s) ⌚ 16:35:02.753760 

AVAILABLE MODEL DEPLOYMENTS (Universal LLM API)

📦 Deployment: gpt-4.1
   ID: aif-6dm44zbv4ms4s-0
   Type: ai-foundry
   SKU: GlobalStandard (Capacity: 100)
   Model: gpt-4.1 (Format: OpenAI, Version: 2025-04-14)
   Capabilities: chatCompletion
   Status: Succeeded

📦 Deployment: DeepSeek-R1
   ID: aif-6dm44zbv4ms4s-0
   Type: ai-found

---
## 🧪 Test Deployed Models

The following sections test the deployed models through both the Universal LLM API and Azure OpenAI API endpoints.
---

<a id='test-universal'></a>
### 🧪 Test via Universal LLM API (models/chat/completions)

Test the deployed models using the Universal LLM API which routes based on the `model` field in the request body:

> **Note:** Some models may not support the chat/completions format. Please refer to your LLM backend documentation for supported API formats per model. There is a validation notebook that is designed to test both chat/completions, embeddings, and response formats.

In [17]:
# Discover the Universal LLM API endpoint
apimClientTool.discover_api("models")
azure_endpoint_models = str(apimClientTool.azure_endpoint)
chat_completions_url_models = f"{azure_endpoint_models}models/chat/completions?api-version={inference_api_version}"

utils.print_info(f"Universal LLM API Endpoint: {chat_completions_url_models}")

# Get an API key from subscriptions (it will get the most recently created Access Contract which should be the one we created for testing)
if apimClientTool.apim_subscriptions:
    api_key = apimClientTool.apim_subscriptions[0].get("key")
    utils.print_ok(f"Using subscription: {apimClientTool.apim_subscriptions[0].get('name')}")
else:
    utils.print_error("No APIM subscriptions found. Please create a subscription first.")
    api_key = None

👉🏽 Found API with id /subscriptions/b881797f-b05b-4743-8516-17a967f31841/resourceGroups/rg-citadel-workshop/providers/Microsoft.ApiManagement/service/apim-6dm44zbv4ms4s/apis/universal-llm-api and path models
👉🏽 Azure Endpoint with APIM https://apim-6dm44zbv4ms4s.azure-api.net/
👉🏽 Universal LLM API Endpoint: https://apim-6dm44zbv4ms4s.azure-api.net/models/chat/completions?api-version=2024-05-01-preview
✅ Using subscription: master ⌚ 17:20:38.512889 


In [18]:
# Test each supported model via Universal LLM API
if api_key and updated_supported_models:
    utils.print_info(f"\nTesting {len(updated_supported_models)} models via Universal LLM API...\n")
    
    test_messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "What is 2+2? Answer in one word."}
    ]
    
    for model_name in updated_supported_models:  # add [:3]:  to test first 3 models only
        utils.print_info(f"Testing model: {model_name}")
        
        payload = {
            "model": model_name,
            "messages": test_messages
        }
        
        try:
            response = requests.post(
                chat_completions_url_models,
                headers={"api-key": api_key},
                json=payload,
                timeout=60
            )
            
            utils.print_response_code(response)
            
            if response.status_code == 200:
                data = response.json()
                answer = data.get("choices", [{}])[0].get("message", {}).get("content", "No response")
                region = response.headers.get("x-ms-region", "unknown")
                print(f"  💬 Response: {answer}")
                print(f"  📍 Backend Region: {region}")
                utils.print_ok(f"Model '{model_name}' - SUCCESS\n")
            else:
                utils.print_error(f"Model '{model_name}' - FAILED: {response.text}\n")
                
        except Exception as e:
            utils.print_error(f"Model '{model_name}' - ERROR: {str(e)}\n")
else:
    utils.print_warning("Cannot run tests - missing API key or supported models")

👉🏽 
Testing 7 models via Universal LLM API...

👉🏽 Testing model: DeepSeek-R1
Response status: 200 - OK
  💬 Response: <think>
Okay, the user asked, "What is 2+2? Answer in one word." Let me think. The question is straightforward. They want the sum of 2 and 2. The answer is 4. They specified to answer in one word, so just the number. I should make sure there's no extra text. Maybe they're testing if I can follow instructions. Yep, "4" is the correct and concise response.
</think>

4
  📍 Backend Region: Sweden Central
✅ Model 'DeepSeek-R1' - SUCCESS
 ⌚ 17:24:38.524543 
👉🏽 Testing model: Mistral-Large-3
Response status: 200 - OK
  💬 Response: Four.
  📍 Backend Region: Sweden Central
✅ Model 'Mistral-Large-3' - SUCCESS
 ⌚ 17:24:45.789706 
👉🏽 Testing model: Phi-4
Response status: 200 - OK
  💬 Response: Four.
  📍 Backend Region: Sweden Central
✅ Model 'Phi-4' - SUCCESS
 ⌚ 17:24:47.297509 
👉🏽 Testing model: gpt-4.1
Response status: 200 - OK
  💬 Response: Four
  📍 Backend Region: Sweden Central

<a id='test-openai'></a>
### 🧪 Test via Azure OpenAI API (openai/deployments/{model}/chat/completions)

Test the deployed models using the Azure OpenAI compatible API which uses the deployment name in the URL path:

In [19]:
# Discover the Azure OpenAI API endpoint
try:
    apimClientTool.discover_api("openai")
    azure_endpoint_openai = str(apimClientTool.azure_endpoint)
    utils.print_info(f"Azure OpenAI API Base Endpoint: {azure_endpoint_openai}")
except Exception as e:
    utils.print_warning(f"Azure OpenAI API not found in APIM: {e}")
    azure_endpoint_openai = None

👉🏽 Found API with id /subscriptions/b881797f-b05b-4743-8516-17a967f31841/resourceGroups/rg-citadel-workshop/providers/Microsoft.ApiManagement/service/apim-6dm44zbv4ms4s/apis/azure-openai-api and path openai
👉🏽 Azure Endpoint with APIM https://apim-6dm44zbv4ms4s.azure-api.net/
👉🏽 Azure OpenAI API Base Endpoint: https://apim-6dm44zbv4ms4s.azure-api.net/


In [20]:
# Test models via Azure OpenAI API format
if api_key and azure_endpoint_openai and updated_supported_models:
    utils.print_info(f"\nTesting models via Azure OpenAI API format...\n")
    
    test_messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "What is the capital of France? Answer in one word."}
    ]
    
    for model_name in updated_supported_models:  # add [:3]:  to test first 3 models only
        utils.print_info(f"Testing model: {model_name}")
        
        # Azure OpenAI format uses deployment name in URL path
        chat_completions_url_openai = f"{azure_endpoint_openai}openai/deployments/{model_name}/chat/completions?api-version={inference_api_version}"
        
        payload = {
            "messages": test_messages  # No model field needed - it's in the URL
        }
        
        try:
            response = requests.post(
                chat_completions_url_openai,
                headers={"api-key": api_key},
                json=payload,
                timeout=60
            )
            
            utils.print_response_code(response)
            
            if response.status_code == 200:
                data = response.json()
                answer = data.get("choices", [{}])[0].get("message", {}).get("content", "No response")
                region = response.headers.get("x-ms-region", "unknown")
                print(f"  💬 Response: {answer}")
                print(f"  📍 Backend Region: {region}")
                utils.print_ok(f"Model '{model_name}' - SUCCESS\n")
            else:
                utils.print_error(f"Model '{model_name}' - FAILED: {response.text}\n")
                
        except Exception as e:
            utils.print_error(f"Model '{model_name}' - ERROR: {str(e)}\n")
else:
    utils.print_warning("Cannot run Azure OpenAI API tests - missing API key, endpoint, or supported models")

👉🏽 
Testing models via Azure OpenAI API format...

👉🏽 Testing model: DeepSeek-R1
Response status: 200 - OK
  💬 Response: <think>
Okay, the user is asking for the capital of France and wants the answer in one word. Let me think. I know that France's capital is Paris. That's straightforward. I should make sure there's no confusion with other cities. No, Paris is definitely the right answer. Just need to confirm and respond with "Paris" without any extra text.
</think>

Paris
  📍 Backend Region: East US 2
✅ Model 'DeepSeek-R1' - SUCCESS
 ⌚ 18:16:33.073272 
👉🏽 Testing model: Mistral-Large-3
Response status: 200 - OK
  💬 Response: Paris.
  📍 Backend Region: Sweden Central
✅ Model 'Mistral-Large-3' - SUCCESS
 ⌚ 18:16:34.383088 
👉🏽 Testing model: Phi-4
Response status: 200 - OK
  💬 Response: Paris.
  📍 Backend Region: East US 2
✅ Model 'Phi-4' - SUCCESS
 ⌚ 18:16:35.542209 
👉🏽 Testing model: gpt-4.1
Response status: 200 - OK
  💬 Response: Paris
  📍 Backend Region: Sweden Central
✅ Model 'gpt-4

<a id='test-sdk'></a>
### 🧪 Test using Azure OpenAI Python SDK

Test using the official Azure OpenAI Python SDK:

In [21]:
from openai import AzureOpenAI
import httpx

def strip_auth_header(request):
    """Remove Authorization header to prevent APIM JWT validation conflict.
    The AzureOpenAI SDK sends both api-key and Authorization: Bearer headers,
    but APIM tries to validate the Authorization header as a JWT token."""
    request.headers.pop("authorization", None)

if api_key and azure_endpoint_openai and updated_supported_models:
    model_name = updated_supported_models[0]  # Use first available model
    utils.print_info(f"Testing with Azure OpenAI SDK using model: {model_name}")
    
    try:
        client = AzureOpenAI(
            azure_endpoint=azure_endpoint_openai,
            api_key=api_key,
            api_version=inference_api_version,
            http_client=httpx.Client(
                event_hooks={"request": [strip_auth_header]}
            )
        )
        
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": "Say 'Hello from Azure OpenAI SDK!'"}
            ]
        )
        
        utils.print_ok("SDK Test Successful!")
        print(f"💬 Response: {response.choices[0].message.content}")
        print(f"📊 Usage: {response.usage.total_tokens} tokens")
        
    except Exception as e:
        utils.print_error(f"SDK Test Failed: {str(e)}")
else:
    utils.print_warning("Cannot run SDK test - missing prerequisites")

👉🏽 Testing with Azure OpenAI SDK using model: DeepSeek-R1
✅ SDK Test Successful! ⌚ 18:20:11.729877 
💬 Response: <think>
Okay, the user wants me to say "Hello from Azure OpenAI SDK!". Let me make sure I get that right.

First, I need to confirm that the exact phrase is "Hello from Azure OpenAI SDK!" with the exclamation mark. No typos here. 

Azure OpenAI SDK refers to the software development kit provided by Azure for their OpenAI services. So the response should be straightforward, just echoing the given message. 

I should check if there's any specific formatting required. The user didn't mention any, so a plain text response is fine. 

No need for additional information unless the user asks for more details. Since the query is simple, just respond with the exact message. 

Alright, ready to send the response.
</think>

Hello from Azure OpenAI SDK!
📊 Usage: 179 tokens


<a id='test-streaming'></a>
### 🧪 Test Streaming Response

Test streaming responses using the Azure OpenAI SDK:

In [22]:
from openai import AzureOpenAI
import httpx

def strip_auth_header(request):
    """Remove Authorization header to prevent APIM JWT validation conflict."""
    request.headers.pop("authorization", None)

if api_key and azure_endpoint_openai and updated_supported_models:
    model_name = updated_supported_models[0]  # Use first available model
    utils.print_info(f"Testing streaming with model: {model_name}")
    
    try:
        client = AzureOpenAI(
            azure_endpoint=azure_endpoint_openai,
            api_key=api_key,
            api_version=inference_api_version,
            http_client=httpx.Client(
                event_hooks={"request": [strip_auth_header]}
            )
        )
        
        start_time = time.time()
        
        response = client.chat.completions.with_raw_response.create(
            model=model_name,
            messages=[
                {"role": "user", "content": "Count from 1 to 10 with commas between each number."}
            ],
            stream=True
        )
        
        print(f"📡 x-ms-region: {response.headers.get('x-ms-region', 'unknown')}")
        print(f"📡 x-ms-stream: {response.headers.get('x-ms-stream', 'N/A')}")
        print("\n💬 Streaming response:")
        
        completion = response.parse()
        collected_content = []
        total_chunks = 0
        content_chunks = 0
        first_chunk_time = None
        
        for chunk in completion:
            total_chunks += 1
            if first_chunk_time is None:
                first_chunk_time = time.time()
            if chunk.choices and chunk.choices[0].delta.content:
                content = chunk.choices[0].delta.content
                collected_content.append(content)
                content_chunks += 1
                print(content, end='', flush=True)
        
        elapsed = time.time() - start_time
        ttfb = (first_chunk_time - start_time) if first_chunk_time else None
        print(f"\n\n✅ Stream completed in {elapsed:.2f} seconds")
        print(f"📦 Chunks received: {total_chunks} (with content: {content_chunks})")
        if ttfb is not None:
            print(f"⚡ Time to first chunk: {ttfb:.2f} seconds")
        print(f"📝 Full response: {''.join(collected_content)}")
        
    except Exception as e:
        utils.print_error(f"Streaming Test Failed: {str(e)}")
else:
    utils.print_warning("Cannot run streaming test - missing prerequisites")


👉🏽 Testing streaming with model: DeepSeek-R1
📡 x-ms-region: Sweden Central
📡 x-ms-stream: N/A

💬 Streaming response:
<think>
Okay, the user wants me to count from 1 to 10 with commas between each number. Let me start by recalling the numbers from 1 to 10. That's straightforward: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10. Wait, but I need to make sure each number is separated by a comma. Let me check each number.

Starting with 1, then a comma. Then 2, comma, 3, comma, and so on. But when I get to 10, there's no number after it, so the last comma isn't needed. So the sequence should be 1, 2, 3, 4, 5, 6, 7, 8, 9, 10. Let me count the numbers to ensure there are ten. 1 through 10 is ten numbers. Each separated by a comma. Let me write them out again: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10. Yep, that looks right. No mistakes here. Just commas between each, and the last number ends without a comma. That should be the correct answer.
</think>

1, 2, 3, 4, 5, 6, 7, 8, 9, 10

✅ Stream completed in 9.18 seconds
📦 C

---
## 📊 Results Summary
---

This notebook completed the following tasks:

1. **Extracted** current APIM backend-pools configurations
2. **Generated** a customizable LLM backend parameter file (`.bicepparam`) supporting:
   - Backend types: `ai-foundry`, `azure-openai`, `aws-bedrock`, `aws-bedrock-mantle`, `gemini-openai`, `external`
   - Auth types: `managed-identity`, `aws-sigv4`, `api-key-bearer`, `api-key-header`, `none`
   - `authConfig` with optional **Key Vault references** (`keyVaultSecretUri`) for API-key backends
   - Per-model metadata (`sku`, `capacity`, `modelFormat`, `modelVersion`, `retirementDate`, `apiVersion`, `timeout`, `inferenceApiVersion`)
   - **Model aliases** (`modelAliases`) with `priority` or `weighted` strategies
   - **AWS credentials** (`awsAccessKey`, `awsSecretKey`, `awsRegion`) for Bedrock
3. **Deployed** the LLM onboarding Bicep templates
4. **Verified** deployment including the `get-available-models` policy fragment
5. **Tested** the deployed models through:
   - **GET /deployments** (Microsoft Foundry integration — uses `get-available-models` policy fragment)
   - Universal LLM API (`/models/chat/completions`)
   - Azure OpenAI API (`/openai/deployments/{model}/chat/completions`)
   - Azure OpenAI Python SDK
   - Streaming responses

### Next Steps

- Modify the `llm_backends_config` in the initialization cell to add more backends (Azure OpenAI, Bedrock, Bedrock Mantle, Gemini, external)
- For API-key backends, set `authConfig.keyVaultSecretUri` and `key_vault_name` to source secrets from Azure Key Vault (recommended over inline `secretValue`)
- Add `model_aliases` to expose simplified client-facing names that route to one or more underlying models
- Provide `aws_access_key` / `aws_secret_key` / `aws_region` when onboarding `aws-bedrock` (SigV4) backends
- Re-run the deployment cells to update the APIM configuration
- Use the generated parameter file as a template for CI/CD pipelines
- Use the `get-available-models` policy fragment to expose available models to Microsoft Foundry and other clients


<a id='cleanup'></a>
### 🧹 Cleanup (Optional)

The backends and policy fragments deployed by this notebook are intended to persist as part of your Governance Hub configuration. To remove them, delete the corresponding backend resources and policy fragments from your APIM instance via the Azure portal or CLI.

> **Note:** Removing backends will affect any access contracts that reference those backend pools. Ensure no active contracts depend on them before cleanup.